# 06 — Program basis and high-level analyses reports

anchor-op works in a low-dimensional program basis, not in gene space. This tutorial covers:

1. Fitting a basis from expression data (`ao.fit_programs`, NMF / cNMF / PCA)
2. Wrapping an externally-computed basis (`ao.make_program_basis`)
3. Projecting expression into program coordinates (`ao.project_expression`)
4. The `ao.analyses.*_report` high-level shortcuts that run measurement + diagnostics + figure
   generation in one call.

In [ ]:
import numpy as np
import pandas as pd
from types import SimpleNamespace
import anchorop as ao
rng = np.random.default_rng(0)

## 1. Fit a program basis from controls

`ao.fit_programs` provides a dependency-light seed-ensemble NMF (`method="cnmf"`) and PCA
(`method="pca"`) as built-in options.

In [ ]:
# Simulate a small control-only expression matrix
n_cells, n_genes = 300, 100
X_ctrl = np.abs(rng.normal(0, 1, size=(n_cells, n_genes)))  # non-negative for NMF

# Package as an AnnData-like object (anchor-op accepts any object with .X, .obs, .var_names)
adata_ctrl = SimpleNamespace(
    X=X_ctrl,
    obs=pd.DataFrame({"is_control": [True] * n_cells}),
    var=pd.DataFrame({"highly_variable": [True] * n_genes},
                     index=[f"g{i}" for i in range(n_genes)]),
    var_names=np.array([f"g{i}" for i in range(n_genes)]),
    layers={},
)

basis_nmf = ao.fit_programs(
    adata_ctrl, d=6, method="cnmf",
    control_mask=adata_ctrl.obs["is_control"].to_numpy(),
    n_seeds=3, seed=0, max_iter=100,
)
print(f"cNMF basis: d = {basis_nmf.d}, control_count = {basis_nmf.control_count}")
print(f"seed_concordance = {basis_nmf.seed_concordance:.3f}  (higher = more stable across seeds)")

## 2. Wrap an external basis

If you've computed a basis with a different tool (Seurat, Scanpy PCA, cNMF standalone), wrap it
with `ao.make_program_basis`.

In [ ]:
# Suppose you have gene × program loadings from an external tool
external_loadings = rng.normal(size=(n_genes, 6)).astype(np.float32)
basis_ext = ao.make_program_basis(
    external_loadings,
    gene_names=adata_ctrl.var_names,
    method="external",
    control_count=n_cells,
    normalize=False,
    metadata={"source": "my_custom_pipeline_v1.2"},
)
print(f"external basis: d = {basis_ext.d}, method = {basis_ext.method}")
print(f"metadata: {basis_ext.metadata}")

## 3. Project expression into program coordinates

`ao.project_expression` takes a (n_cells × n_genes) expression matrix and a basis, returns
(n_cells × d) program coordinates.

In [ ]:
z_ctrl = ao.project_expression(X_ctrl, basis_ext, gene_names=adata_ctrl.var_names)
print(f"program coordinates shape: {z_ctrl.shape}")
print(f"mean per program: {z_ctrl.mean(axis=0)}")

## 4. High-level analyses reports

The `ao.analyses` submodule wraps common workflows into one-call shortcuts that return standard
figures + JSON/CSV summaries.

**`measurement_report(measurement, save_dir=...)`** — diagnostic panel (singular spectrum, operator
heatmap, eigenvalue plane) + guide-drop pareto + summary JSON. This is the wrapper used by the
paper's Fig 3, Fig 4.

**`benchmark_report(measurement, inferred_dict, save_dir=...)`** — see tutorial 05.

**`archetype_report(measurements_dict, ks, save_dir=...)`** — see tutorial 05.

In [ ]:
# Build a small measurement to demo measurement_report
d, n = 6, 30
J_true = (rng.normal(size=(d, d)) / np.sqrt(d)) - 2 * np.eye(d)
Wd = rng.normal(size=(d, n)); Wd /= np.linalg.norm(Wd, axis=0, keepdims=True)
kappa = 0.3 + 0.6 * rng.uniform(size=n)
U = -kappa[None, :] * Wd
S = -np.linalg.solve(J_true, U) + 0.02 * rng.normal(size=(d, n))
names = [f"g{i}" for i in range(n)]
m = ao.measure_from_sensitivity(S, U, guide_names=names,
                                  guide_efficiencies={nm: float(kappa[i]) for i, nm in enumerate(names)},
                                  reg="tsvd", reg_param="path", rank_tol=1e-2)

report = ao.analyses.measurement_report(m, save_dir=None)   # or save_dir="path/to/output"
print("report keys:", list(report.keys()))
print("figure names:", list(report["figures"].keys()))
print("summary:", report["summary"])

### Passing `save_dir=`

When you supply `save_dir`, the report writes:

- `diagnostics.png` — 3-panel diagnostic figure
- `guide_drops.png` — guide-drop pareto
- `summary.json` — numerical fields (rank, condition, retained/dropped counts, etc.)

This is the direct code path for reproducing Fig 3a/b and Fig 4a/b in the manuscript.

## 5. IO helpers

Persist and load measurements + reports:

In [ ]:
# Save a measurement (uses pickle under the hood)
# ao.io.save_operator(m, "path/to/measurement.pkl")   # or use pickle directly
# loaded = ao.load_operator("path/to/measurement.pkl")
# valid = ao.validate_operator(loaded)                # sanity checks
print("(IO helpers ao.load_operator / ao.validate_operator are for round-tripping saved operators.)")

## 6. Replogle-aware loader

For Replogle 2022 Perturb-seq h5ads (essential-gene or genome-scale), `ao.load_replogle_h5ad`
auto-detects the guide / target / batch column names, which vary across releases, and returns
an AnnData with canonical `obs["guide"]` and `obs["target_gene"]` columns ready for
`measure_operator`. This is exactly what the paper's `reproduction/03_fig3_k562_essential.py`
and `04_fig4_rpe1_essential.py` use.

In [ ]:
# adata = ao.load_replogle_h5ad("path/to/K562_essential_normalized_singlecell_01.h5ad")
# # Now adata.obs has canonical 'guide' and 'target_gene' columns
# m = ao.measure_operator(adata, basis, guide_key="guide", target_key="target_gene",
#                          control_label="non-targeting", ...)
print("(load_replogle_h5ad handles schema variation across Replogle releases automatically.)")

## Summary

You now know every public entry point in anchor-op:

- **Data loading**: `load_replogle_h5ad`
- **Program basis**: `fit_programs`, `make_program_basis`, `project_expression`
- **Efficiency estimation**: `estimate_knockdown_efficiency{,_detection_rate,_poisson_mle}`,
  the `"auto"` router (tutorial 02)
- **Measurement**: `measure_operator`, `measure_from_sensitivity`, `build_guide_responses`
  (tutorial 03)
- **Identifiability**: `regularization_path`, `regularized_pseudoinverse`, and the AnchorReport
  fields (tutorial 03)
- **Linearity diagnostics**: `linearity_check`, `held_out_prediction_check`, with matched-scale
  positive controls (tutorial 04)
- **Comparison and benchmark**: `compare`, `comparison_table`, `spectral_*`, `analyses.benchmark_report`
  (tutorial 05)
- **Archetypes**: `fit_archetypes`, `transfer_test`, `analyses.archetype_report` (tutorial 05)
- **High-level reports**: `analyses.measurement_report`, `analyses.benchmark_report`,
  `analyses.archetype_report`

For per-figure reproducibility of the manuscript, see `../reproduction/`.